# Deep Tabular Networks & Graph Neural Networks

This notebook covers:
1. **Tabular Deep Learning**: TabNet with Gated Linear Units (GLU), Wide & Deep networks
2. **Graph Neural Networks**: GCN and GAT from scratch with PyTorch
3. **Graph Operations**: Sparse matrix operations, adjacency processing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.sparse as sp
import numpy as np
from scipy import sparse as scipy_sparse

# =====================================================================
# GATED LINEAR UNIT (GLU) FOR TABULAR DATA
# =====================================================================

class GLUBlock(nn.Module):
    """
    Gated Linear Unit: Feature Gating Mechanism
    Output = (Linear1(x)) * sigmoid(Linear2(x))
    Learns which features are important via gating.
    """
    
    def __init__(self, input_dim, output_dim, virtual_batch_size=128, dropout=0.0):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.virtual_batch_size = virtual_batch_size
        
        # Linear transformations
        self.fc = nn.Linear(input_dim, output_dim)
        self.gate = nn.Linear(input_dim, output_dim)
        
        self.bn = nn.BatchNorm1d(output_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, input_dim]
        Returns:
            gated_output: [batch_size, output_dim]
        """
        # Compute feature representation and gate
        linear_out = self.fc(x)  # [batch, output_dim]
        gate_out = torch.sigmoid(self.gate(x))  # [batch, output_dim]
        
        # Gating: element-wise multiplication
        gated = linear_out * gate_out
        
        # Batch normalization
        gated = self.bn(gated)
        gated = self.dropout(gated)
        
        return gated


class TabNet(nn.Module):
    """
    TabNet: Tabular Data with Sequential Attention.
    Uses GLU blocks with feature masking for interpretability.
    """
    
    def __init__(self, input_dim, n_classes, n_steps=3, n_independent=2, 
                 n_shared=2, dim=64, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.n_steps = n_steps
        self.n_independent = n_independent
        self.n_shared = n_shared
        
        # Initial processing
        self.initial_bn = nn.BatchNorm1d(input_dim)
        
        # Shared GLU layers
        self.shared = nn.ModuleList()
        for i in range(n_shared):
            self.shared.append(GLUBlock(input_dim if i == 0 else dim, dim, dropout=dropout))
        
        # Step-wise GLU blocks
        self.step_blocks = nn.ModuleList()
        for step in range(n_steps):
            step_glu = nn.ModuleList()
            for i in range(n_independent):
                if i == 0:
                    step_glu.append(GLUBlock(dim, dim, dropout=dropout))
                else:
                    step_glu.append(GLUBlock(dim, dim, dropout=dropout))
            self.step_blocks.append(step_glu)
        
        # Final classification head
        self.final_fc = nn.Linear(dim, n_classes)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, input_dim]
        Returns:
            logits: [batch_size, n_classes]
        """
        x = self.initial_bn(x)
        
        # Shared layers
        for layer in self.shared:
            x = layer(x)
        
        # Sequential decision steps
        for step_layers in self.step_blocks:
            for layer in step_layers:
                x = layer(x) + x  # Residual connection
        
        # Classification
        logits = self.final_fc(x)
        return logits


In [ ]:
# =====================================================================
# WIDE & DEEP NETWORK
# =====================================================================

class WideDeepNetwork(nn.Module):
    """
    Wide & Deep Learning (Google, 2016).
    Combines:
    - Wide: Direct feature combinations (memory of specific past patterns)
    - Deep: Non-linear feature interactions (generalization)
    """
    
    def __init__(self, input_dim, embedding_dim=16, hidden_dims=[256, 128], 
                 n_classes=1, dropout=0.1):
        super().__init__()
        
        # WIDE PART: Linear transformation with cross-features
        self.wide = nn.Linear(input_dim, n_classes)
        
        # DEEP PART: Multi-layer network
        deep_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            deep_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        # Final layer in deep part
        deep_layers.append(nn.Linear(prev_dim, embedding_dim))
        self.deep = nn.Sequential(*deep_layers)
        
        # FINAL COMBINATION
        self.final = nn.Linear(embedding_dim, n_classes)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, input_dim]
        Returns:
            logits: [batch_size, n_classes]
        """
        # Wide path
        wide_out = self.wide(x)  # [batch, n_classes]
        
        # Deep path
        deep_out = self.deep(x)  # [batch, embedding_dim]
        deep_logits = self.final(deep_out)  # [batch, n_classes]
        
        # Combine wide and deep
        logits = wide_out + deep_logits
        
        return logits


In [ ]:
# =====================================================================
# GRAPH NEURAL NETWORKS: GCN (GRAPH CONVOLUTIONAL NETWORK)
# =====================================================================

class GraphConvolution(nn.Module):
    """
    Graph Convolutional Layer.
    H' = D^(-1/2) * A_hat * D^(-1/2) * H * W
    where A_hat = A + I (adjacency + self-loops)
    and D is the degree matrix.
    """
    
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Weight matrix
        self.weight = nn.Parameter(torch.Tensor(in_features, out_features))
        
        if bias:
            self.bias = nn.Parameter(torch.Tensor(out_features))
        else:
            self.register_parameter('bias', None)
        
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        """
        Args:
            x: [n_nodes, in_features] - Node features
            adj: [n_nodes, n_nodes] - Adjacency matrix (dense or sparse)
        Returns:
            output: [n_nodes, out_features]
        """
        # Matrix multiplication: adj @ x @ W
        support = torch.mm(x, self.weight)  # [n_nodes, out_features]
        output = torch.sparse.mm(adj, support) if adj.is_sparse else torch.mm(adj, support)
        
        if self.bias is not None:
            output = output + self.bias
        
        return output


class GCN(nn.Module):
    """
    Graph Convolutional Network.
    For node classification on graphs.
    """
    
    def __init__(self, n_features, n_hidden, n_classes, n_layers=2, dropout=0.5):
        super().__init__()
        self.n_features = n_features
        self.n_hidden = n_hidden
        self.n_classes = n_classes
        self.dropout = dropout
        
        # Graph convolutional layers
        self.gc_layers = nn.ModuleList()
        self.gc_layers.append(GraphConvolution(n_features, n_hidden))
        
        for _ in range(n_layers - 2):
            self.gc_layers.append(GraphConvolution(n_hidden, n_hidden))
        
        self.gc_layers.append(GraphConvolution(n_hidden, n_classes))
        
        self.dropout_layer = nn.Dropout(dropout)
    
    def forward(self, x, adj):
        """
        Args:
            x: [n_nodes, n_features] - Node features
            adj: [n_nodes, n_nodes] - Normalized adjacency matrix
        Returns:
            logits: [n_nodes, n_classes]
        """
        for i, gc_layer in enumerate(self.gc_layers[:-1]):
            x = gc_layer(x, adj)
            x = F.relu(x)
            x = self.dropout_layer(x)
        
        # Final layer (no activation)
        x = self.gc_layers[-1](x, adj)
        
        return x


In [ ]:
# =====================================================================
# GRAPH NEURAL NETWORKS: GAT (GRAPH ATTENTION NETWORK)
# =====================================================================

class GraphAttentionLayer(nn.Module):
    """
    Graph Attention Layer.
    Uses self-attention to compute edge weights based on node features.
    α_ij = softmax(LeakyReLU(a^T [W*h_i || W*h_j]))
    h'_i = σ(Σ_j α_ij * W * h_j)
    """
    
    def __init__(self, in_features, out_features, n_heads=8, dropout=0.6, 
                 concat=True, negative_slope=0.2):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_heads = n_heads
        self.dropout = dropout
        self.concat = concat
        self.negative_slope = negative_slope
        
        # Multi-head attention
        assert out_features % n_heads == 0, "out_features must be divisible by n_heads"
        self.head_dim = out_features // n_heads
        
        # Linear transformation for each head
        self.W = nn.Linear(in_features, out_features, bias=False)
        
        # Attention coefficients
        self.a = nn.Parameter(torch.Tensor(1, n_heads, 2 * self.head_dim))
        
        # Bias
        self.bias = nn.Parameter(torch.Tensor(out_features))
        
        # Dropout
        self.dropout_layer = nn.Dropout(dropout)
        
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        """
        Args:
            x: [n_nodes, in_features]
            adj: [n_nodes, n_nodes] - Binary adjacency matrix
        Returns:
            output: [n_nodes, out_features]
        """
        n_nodes = x.size(0)
        
        # Linear transformation
        h = self.W(x)  # [n_nodes, out_features]
        h = h.view(n_nodes, self.n_heads, self.head_dim)
        
        # Pairwise attention scores
        # Create all pairs: [n_nodes, n_nodes, 2*head_dim]
        h_i = h.unsqueeze(1).expand(n_nodes, n_nodes, self.n_heads, self.head_dim)
        h_j = h.unsqueeze(0).expand(n_nodes, n_nodes, self.n_heads, self.head_dim)
        
        # Concatenate h_i and h_j
        h_ij = torch.cat([h_i, h_j], dim=-1)  # [n_nodes, n_nodes, n_heads, 2*head_dim]
        
        # Attention scores: [n_nodes, n_nodes, n_heads]
        scores = torch.einsum('nijh,hd->nij', h_ij, self.a.squeeze(0))
        scores = F.leaky_relu(scores, negative_slope=self.negative_slope)
        
        # Mask: set scores to -inf where there's no edge
        adj_expanded = adj.unsqueeze(2).expand(-1, -1, self.n_heads)
        scores = scores.masked_fill(adj_expanded == 0, -1e9)
        
        # Softmax attention weights
        alpha = F.softmax(scores, dim=1)  # [n_nodes, n_nodes, n_heads]
        alpha = self.dropout_layer(alpha)
        
        # Apply attention weights
        # alpha: [n_nodes, n_nodes, n_heads]
        # h_j: [n_nodes, n_nodes, n_heads, head_dim]
        output = torch.einsum('nijh,nijhd->ihd', alpha, h_j)  # [n_nodes, n_heads, head_dim]
        output = output.reshape(n_nodes, -1)  # [n_nodes, out_features]
        
        # Bias
        output = output + self.bias
        
        return output


class GAT(nn.Module):
    """
    Graph Attention Network.
    Uses multi-head attention to compute node representations.
    """
    
    def __init__(self, n_features, n_hidden, n_classes, n_heads=8, n_layers=2, 
                 dropout=0.6, negative_slope=0.2):
        super().__init__()
        self.n_features = n_features
        self.n_hidden = n_hidden
        self.n_classes = n_classes
        self.n_heads = n_heads
        
        # Attention layers
        self.att_layers = nn.ModuleList()
        self.att_layers.append(
            GraphAttentionLayer(n_features, n_hidden, n_heads, dropout, True, negative_slope)
        )
        
        for _ in range(n_layers - 2):
            self.att_layers.append(
                GraphAttentionLayer(n_hidden, n_hidden, n_heads, dropout, True, negative_slope)
            )
        
        self.att_layers.append(
            GraphAttentionLayer(n_hidden, n_classes, 1, dropout, False, negative_slope)
        )
        
        self.dropout_layer = nn.Dropout(dropout)
    
    def forward(self, x, adj):
        """
        Args:
            x: [n_nodes, n_features]
            adj: [n_nodes, n_nodes] - Binary adjacency matrix
        Returns:
            logits: [n_nodes, n_classes]
        """
        for i, att_layer in enumerate(self.att_layers[:-1]):
            x = att_layer(x, adj)
            x = F.relu(x)
            x = self.dropout_layer(x)
        
        # Final layer (no activation)
        x = self.att_layers[-1](x, adj)
        
        return x


In [ ]:
# =====================================================================
# GRAPH UTILITY FUNCTIONS
# =====================================================================

def normalize_adjacency_matrix(adj):
    """
    Normalize adjacency matrix: D^(-1/2) * A_hat * D^(-1/2)
    where A_hat = A + I (self-loops added)
    
    Args:
        adj: [n_nodes, n_nodes] sparse or dense adjacency matrix
    Returns:
        normalized_adj: Normalized adjacency matrix
    """
    # Add self-loops
    if adj.is_sparse:
        indices = adj.indices()
        values = adj.values()
        n_nodes = adj.size(0)
        # Convert to dense for simplicity
        adj = adj.to_dense()
    else:
        n_nodes = adj.size(0)
    
    # Add identity (self-loops)
    adj_with_self_loops = adj + torch.eye(n_nodes, device=adj.device)
    
    # Compute degree matrix
    degree = adj_with_self_loops.sum(dim=1)  # [n_nodes]
    degree_inv_sqrt = torch.pow(degree, -0.5)
    degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0.0
    
    # D^(-1/2)
    D_inv_sqrt = torch.diag(degree_inv_sqrt)
    
    # D^(-1/2) * A_hat * D^(-1/2)
    normalized = D_inv_sqrt @ adj_with_self_loops @ D_inv_sqrt
    
    return normalized


def create_adjacency_from_edge_list(edges, n_nodes):
    """
    Create adjacency matrix from edge list.
    
    Args:
        edges: [n_edges, 2] tensor with edge indices
        n_nodes: Total number of nodes
    Returns:
        adj: [n_nodes, n_nodes] sparse adjacency matrix
    """
    # Create sparse adjacency matrix
    indices = edges.t().contiguous()
    values = torch.ones(edges.size(0), device=edges.device)
    
    # Make undirected
    indices_rev = torch.stack([indices[1], indices[0]], dim=0)
    indices_full = torch.cat([indices, indices_rev], dim=1)
    values_full = torch.cat([values, values])
    
    adj = torch.sparse_coo_tensor(indices_full, values_full, (n_nodes, n_nodes))
    
    return adj


In [ ]:
# =====================================================================
# DEMONSTRATION AND TESTING
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test 1: TabNet
    print("=" * 60)
    print("TABNET: GATED LINEAR UNITS FOR TABULAR DATA")
    print("=" * 60)
    
    tabnet = TabNet(input_dim=50, n_classes=10, n_steps=3).to(device)
    x_tabular = torch.randn(16, 50).to(device)
    
    tabnet.eval()
    with torch.no_grad():
        logits = tabnet(x_tabular)
    
    print(f"Input shape: {x_tabular.shape}")
    print(f"Output shape: {logits.shape}")
    print(f"✓ TabNet working!\n")
    
    # Test 2: Wide & Deep
    print("=" * 60)
    print("WIDE & DEEP NETWORK")
    print("=" * 60)
    
    wide_deep = WideDeepNetwork(input_dim=50, n_classes=10).to(device)
    
    wide_deep.eval()
    with torch.no_grad():
        logits = wide_deep(x_tabular)
    
    print(f"Output shape: {logits.shape}")
    print(f"✓ Wide & Deep working!\n")
    
    # Test 3: GCN
    print("=" * 60)
    print("GRAPH CONVOLUTIONAL NETWORK (GCN)")
    print("=" * 60)
    
    n_nodes = 20
    n_features = 16
    n_classes = 4
    
    # Create random graph
    adj = torch.randint(0, 2, (n_nodes, n_nodes)).float()
    adj = (adj + adj.t()) / 2  # Make symmetric
    adj.fill_diagonal_(0)  # Remove self-loops (will be added during normalization)
    
    # Normalize adjacency
    adj_norm = normalize_adjacency_matrix(adj)
    
    # Node features
    x_graph = torch.randn(n_nodes, n_features)
    
    gcn = GCN(n_features=n_features, n_hidden=32, n_classes=n_classes).to(device)
    adj_norm = adj_norm.to(device)
    x_graph = x_graph.to(device)
    
    gcn.eval()
    with torch.no_grad():
        logits = gcn(x_graph, adj_norm)
    
    print(f"Node features shape: {x_graph.shape}")
    print(f"Adjacency shape: {adj_norm.shape}")
    print(f"GCN output shape: {logits.shape}")
    print(f"✓ GCN working!\n")
    
    # Test 4: GAT
    print("=" * 60)
    print("GRAPH ATTENTION NETWORK (GAT)")
    print("=" * 60)
    
    gat = GAT(n_features=n_features, n_hidden=32, n_classes=n_classes, n_heads=4).to(device)
    
    # Convert adjacency to binary for attention
    adj_binary = (adj > 0).float().to(device)
    
    gat.eval()
    with torch.no_grad():
        logits = gat(x_graph, adj_binary)
    
    print(f"GAT output shape: {logits.shape}")
    print(f"✓ GAT working!\n")
    
    print("✅ All tabular and graph models working correctly!")
